[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/entropy_production/entropy_production.ipynb)

# Entropy Production in Heat Exchange

This example computes how two insulated bodies at different temperatures share their energy until their temperatures meet. Heat flows from the hotter body to the colder one, which keeps the total energy fixed and produces entropy, because the heat leaves at a higher temperature than it arrives at; the bodies come to rest at the state of most entropy their total energy allows.

A state is modelled as the two bodies' energies, an antivector of the plane spanned by `mv.a.dual()` and `mv.b.dual()`. The entropy's covector, each body's received heat over its temperature, `mv.a / temperature_a + mv.b / temperature_b`, pairs with a change of state to the entropy it produces. The mobility is a map from covectors to changes of state, a dyad, `transfer * (Covector & transfer)` times a positive strength: it sends every covector into the direction `transfer` that moves energy from one body to the other. It sends the entropy's covector to the heat flow, and the total energy's covector, `mv.a + mv.b`, which reads `transfer` as nothing, to nothing; so the flow keeps the energy and, the dyad being positive, never lowers the entropy.

In the usual notation the bodies follow Newton's law of cooling, $C_a \dot T_a = -k(T_a - T_b) = -C_b \dot T_b$, and the entropy is produced at the rate $\dot S = k (T_a - T_b)^2 / (T_a T_b) \ge 0$. In the GENERIC formulation this dissipative part is $\dot x = M\,dS$, with $M$ positive and $M\,dE = 0$.

Energies are in units of the heat capacity times a reference temperature, temperatures in that reference.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import jax
import numpy as np

from numga import Algebra
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.entropy_production import render

jax.config.update("jax_enable_x64", True)
ga = Algebra("a0b0")
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()
State = ga.gatype.antivector()
Covector = ga.gatype.vector()
Mobility = ga.gatype((State, Covector))

# The total energy's covector, and the direction moving energy from body a to body b, which it reads as nothing.
energy_covector = mv.a + mv.b                             # [] Covector
transfer = mv.b.dual() - mv.a.dual()                      # [] State
# The bodies' heat capacities, and the conductance of the bar between them.
capacity_a, capacity_b, conductance = 1.0, 2.0, 0.6

## The entropy, its covector and the mobility

A body's temperature is its energy over its heat capacity, and its entropy the heat capacity times the logarithm of its temperature. The entropy's covector is its derivative as a covector, the received heat over the temperature for each body. The mobility's strength is the conductance times the product of the temperatures, which makes the heat flow the conductance times the temperature difference. Body a starts hot and body b cold; the chart on the left shows the plane of their energies, the entropy's level lines faint and the line of fixed total energy, along which the state can only move, and on the right the entropy along that line.

In [ ]:
def temperatures(state: State) -> tuple[Scalar, Scalar]:
    return (mv.a & state) / capacity_a, (mv.b & state) / capacity_b

def entropy(state: State) -> Scalar:
    temperature_a, temperature_b = temperatures(state)
    return capacity_a * temperature_a.log() + capacity_b * temperature_b.log()

def entropy_covector(state: State) -> Covector:
    """Each body's received heat over its temperature."""
    temperature_a, temperature_b = temperatures(state)
    return mv.a / temperature_a + mv.b / temperature_b

def mobility(state: State, strength: float) -> Mobility:
    """The dyad sending every covector along the transfer, with the strength times the temperatures' product."""
    temperature_a, temperature_b = temperatures(state)
    return transfer * (Covector & transfer) * strength * temperature_a * temperature_b

start = mv.a.dual() * (capacity_a * 2.8) + mv.b.dual() * (capacity_b * 0.6)                 # [] State
heat_flow = mobility(start, conductance)(entropy_covector(start))                           # [] State
capacities = np.array([capacity_a, capacity_b])
render.draw(entropy, temperatures, transfer, capacities, start, heat_flow)

In [ ]:
print("heat flow:", (mv.a & heat_flow).to_array(), (mv.b & heat_flow).to_array())
print("entropy produced per unit time:", (entropy_covector(start) & heat_flow).to_array())
print("energy gained per unit time:", (energy_covector & heat_flow).to_array())

## The exchange over time

The heat flow is the conductance times the temperature difference, and that difference decays exponentially with the relaxation time `1 / (conductance * (1 / capacity_a + 1 / capacity_b))`. So a step of any length is exact as one application of the mobility with the conductance replaced by its integral over the step, `-expm1(-conductance * inverse_capacity * dt) / inverse_capacity`. The animation runs for five relaxation times.

In [ ]:
inverse_capacity = 1 / capacity_a + 1 / capacity_b
relaxation_time = 1 / (conductance * inverse_capacity)
n_frames, n_relaxation_times = 72, 5
dt = n_relaxation_times * relaxation_time / (n_frames - 1)
integrated_conductance = -np.expm1(-conductance * inverse_capacity * dt) / inverse_capacity

def exchange():
    """The bodies' state and its heat flow at each frame."""
    state = start
    for _ in range(n_frames):
        yield state, mobility(state, conductance)(entropy_covector(state))
        state = state + mobility(state, integrated_conductance)(entropy_covector(state))

show_animation(render.exchange(entropy, temperatures, transfer, capacities, exchange()), "entropy_production",
               duration_ms=75)

The hot body a gives energy to body b along the line of fixed total energy, the arrow shrinking as the temperatures approach, and the state climbs the entropy's hill on the right. It comes to rest where the green level line of the entropy touches the line of fixed energy, at the top of the hill, where both bodies are at the common temperature $4/3$, the total energy over the total heat capacity.

In [ ]:
end = list(exchange())[-1][0]                                                               # [] State
final_temperatures = temperatures(end)

In [ ]:
print("final temperatures:", final_temperatures[0].to_array(), final_temperatures[1].to_array())
print("relaxation time:", relaxation_time)

In [ ]:
# checks
states = [state for state, _ in exchange()]
np.testing.assert_allclose([((energy_covector & state) - (energy_covector & start)).to_array() for state in states], 0, atol=1e-10)
assert np.all(np.diff([entropy(state).to_array() for state in states]) >= -1e-12)
# The entropy's covector, as a reading of changes of state, is the entropy's derivative.
np.testing.assert_allclose(((entropy_covector(start) & State) - derivative(entropy)(start)).kernel, 0, atol=1e-10)
np.testing.assert_allclose(mobility(start, conductance)(energy_covector).kernel, 0, atol=1e-10)
np.testing.assert_allclose([temperature.to_array() for temperature in final_temperatures], 4 / 3, atol=1e-2)